# Generate Google Slides Presentation for QM 2023 Capstone Project

This notebook will guide you through creating a Google Slides presentation programmatically using the Google Slides API. Follow each section to authenticate, create, and populate your presentation with content and images.

## 1. Install Required Libraries

Install the Google API Python client and authentication libraries using pip.

In [ ]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

## 2. Import Libraries

Import necessary Python libraries for Google Slides API interaction and authentication.

In [ ]:
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle
import os.path

## 3. Create Google Slides Presentation

Authenticate with Google and create a new Google Slides presentation using the API.

In [ ]:
# Authenticate and create a new Google Slides presentation
SCOPES = ['https://www.googleapis.com/auth/presentations', 'https://www.googleapis.com/auth/drive.file']
creds = None
if os.path.exists('token.pickle'):
    with open('token.pickle', 'rb') as token:
        creds = pickle.load(token)
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
    with open('token.pickle', 'wb') as token:
        pickle.dump(creds, token)
service = build('slides', 'v1', credentials=creds)
presentation = service.presentations().create(body={
    'title': 'QM 2023 Capstone Project: Milestone 4'
}).execute()
presentation_id = presentation.get('presentationId')
print(f'Created presentation with ID: {presentation_id}')

## 4. Add Slides with Titles and Content

Programmatically add slides to the presentation, setting titles and body text for each slide.

In [ ]:
# Example: Add slides with titles and content
def create_slide(service, presentation_id, title, content):
    requests = [
        {
            'createSlide': {
                'slideLayoutReference': {
                    'predefinedLayout': 'TITLE_AND_BODY'
                }
            }
        }
    ]
    response = service.presentations().batchUpdate(
        presentationId=presentation_id, body={'requests': requests}).execute()
    slide_id = response.get('replies')[0]['createSlide']['objectId']
    requests = [
        {
            'insertText': {
                'objectId': slide_id,
                'text': title,
                'insertionIndex': 0
            }
        },
        {
            'insertText': {
                'objectId': slide_id,
                'text': content,
                'insertionIndex': 0
            }
        }
    ]
    service.presentations().batchUpdate(
        presentationId=presentation_id, body={'requests': requests}).execute()

# Example usage:
create_slide(service, presentation_id, 'Executive Summary', 'No meaningful disaster-price penalty found. Macro-financing conditions are stronger drivers. Recommendation: Focus on rate-sensitive markets.')
create_slide(service, presentation_id, 'Methodology', 'Data: NOAA Storm Events, Yale/Shiller, FRED. Panel: 116,137 rows, 3,347 counties, 1980-2022. Model: Two-way fixed effects.')
create_slide(service, presentation_id, 'Results', 'Disaster coefficients: Zero. Robustness: Stable. ARIMA outperforms naive. Random forest > OLS.')
create_slide(service, presentation_id, 'Conclusions & Recommendations', 'Focus on rate-aware allocation. Prioritize strong fundamentals. Disaster risk is secondary.')
create_slide(service, presentation_id, 'References & AI Audit', 'FRED, NOAA, Shiller. Copilot used for code and memo drafting. All results verified.')
create_slide(service, presentation_id, 'Q&A', 'Questions?')

## 5. Add Images to Slides

Insert images into specific slides using image URLs or uploaded files.

In [ ]:
# Example: Add image to a slide (replace IMAGE_URL and slide_id as needed)
def add_image_to_slide(service, presentation_id, slide_id, image_url):
    requests = [
        {
            'createImage': {
                'url': image_url,
                'elementProperties': {
                    'pageObjectId': slide_id,
                    'size': {
                        'height': {'magnitude': 3000000, 'unit': 'EMU'},
                        'width': {'magnitude': 4000000, 'unit': 'EMU'}
                    },
                    'transform': {
                        'scaleX': 1,
                        'scaleY': 1,
                        'translateX': 1000000,
                        'translateY': 1000000,
                        'unit': 'EMU'
                    }
                }
            }
        }
    ]
    service.presentations().batchUpdate(
        presentationId=presentation_id, body={'requests': requests}).execute()

# Example usage (update slide_id and image_url):
# add_image_to_slide(service, presentation_id, slide_id, 'https://example.com/image.png')

## 6. Save and Export the Presentation

Save changes and export or share the Google Slides presentation link.

In [ ]:
# Share the presentation link
print(f'Your Google Slides presentation is ready: https://docs.google.com/presentation/d/{presentation_id}/edit')